# Trabajo Final de Machine Learning

## Preparación y análisis exploratorio de datos financieros

Este notebook presenta el avance inicial de una solución para analizar datos históricos del índice S&P 500. En esta etapa se aplican **Pandas** y **NumPy** para cargar, explorar, limpiar, transformar y preparar la información antes de implementar los modelos predictivos.

## Problemática

Una empresa de análisis financiero maneja grandes volúmenes de datos en archivos CSV, pero no cuenta con un procedimiento eficiente y reproducible para limpiarlos, analizarlos y prepararlos antes de aplicar Machine Learning. Esta situación dificulta la identificación de tendencias y la toma de decisiones basada en datos.

## Objetivo del avance

Preparar un dataset histórico del índice S&P 500 mediante Pandas y NumPy, realizando su ingesta, exploración, limpieza, transformación y análisis descriptivo. El resultado servirá como base para definir las variables independientes (X), la variable objetivo (y) y el modelo de clasificación en el siguiente avance.

### Alcance actual

- Lectura del archivo CSV.
- Exploración de filas, columnas y tipos de datos.
- Detección de valores nulos, duplicados y datos inválidos.
- Conversión y traducción de columnas.
- Creación de indicadores financieros básicos.
- Análisis estadístico con Pandas y NumPy.
- Visualización de distribuciones, evolución y correlaciones.

## 1. Importación de librerías

Se importan las librerías necesarias para manipular, analizar y visualizar los datos.

In [ ]:
# Importa NumPy para realizar operaciones numéricas con vectores y matrices
import numpy as np

# Importa Pandas para trabajar con DataFrames y archivos CSV
import pandas as pd

# Importa Matplotlib y Seaborn para crear visualizaciones
import matplotlib.pyplot as plt
import seaborn as sns

# Configura el tamaño y estilo predeterminados de los gráficos
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_theme(style="whitegrid")

print("Librerías importadas correctamente.")

## 2. Carga del dataset

El archivo se encuentra en la carpeta `data`, mientras que este notebook se ejecuta desde la carpeta `notebooks`. La clase `Path` permite construir y verificar la ruta antes de leer el CSV.

In [ ]:
from pathlib import Path

# Define la ubicación del archivo original
ruta_archivo = Path("../data/snp500_history.csv")

# Detiene la ejecución con un mensaje claro si el archivo no existe
if not ruta_archivo.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo en: {ruta_archivo.resolve()}"
    )

# Lee el CSV y lo convierte en un DataFrame
df = pd.read_csv(ruta_archivo)

print("Archivo cargado:", ruta_archivo.resolve())
print("Dimensiones originales:", df.shape)

## 3. Exploración inicial

La exploración permite conocer la estructura del dataset antes de modificarlo. Cada fila representa una fecha de negociación y cada columna contiene una característica financiera.

In [ ]:
# Muestra una primera muestra del contenido
display(df.head(10))

# Informa la cantidad de filas, columnas y sus nombres
cantidad_filas, cantidad_columnas = df.shape
print("Cantidad de filas:", cantidad_filas)
print("Cantidad de columnas:", cantidad_columnas)
print("Columnas originales:", df.columns.tolist())

In [ ]:
# Muestra los tipos de datos y la cantidad de valores no nulos
df.info()

## 4. Limpieza de datos

Se revisan valores nulos, registros duplicados, columnas sin nombre, fechas repetidas, precios inválidos y columnas que no presentan variación.

In [ ]:
# Cuenta los valores nulos de cada columna
valores_nulos = df.isnull().sum()

if (valores_nulos > 0).any():
    print("Columnas con valores nulos:")
    display(valores_nulos[valores_nulos > 0])
else:
    print("El dataset no contiene valores nulos.")

In [ ]:
# Elimina filas completamente repetidas y registra el resultado
filas_antes = len(df)
df = df.drop_duplicates().copy()
filas_eliminadas = filas_antes - len(df)

print("Filas duplicadas eliminadas:", filas_eliminadas)

In [ ]:
# Elimina columnas auxiliares creadas al exportar otros DataFrames
columnas_sin_nombre = df.columns.str.contains(
    "^Unnamed",
    case=False,
    regex=True
)

df = df.loc[:, ~columnas_sin_nombre].copy()
print("Columnas después de eliminar nombres auxiliares:", df.columns.tolist())

In [ ]:
# Convierte Date de texto a fecha y normaliza la zona horaria
df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce",
    utc=True
).dt.tz_localize(None).dt.normalize()

# Elimina fechas que no pudieron convertirse
fechas_invalidas = df["Date"].isna().sum()
df = df.dropna(subset=["Date"]).copy()

# Ordena cronológicamente y elimina fechas repetidas
fechas_duplicadas = df.duplicated(subset=["Date"]).sum()
df = (
    df.sort_values("Date")
      .drop_duplicates(subset=["Date"], keep="last")
      .reset_index(drop=True)
)

print("Fechas inválidas eliminadas:", fechas_invalidas)
print("Fechas duplicadas eliminadas:", fechas_duplicadas)
print("Tipo de Date:", df["Date"].dtype)

In [ ]:
# Traduce los nombres para facilitar la lectura y exposición
df = df.rename(columns={
    "Date": "Fecha",
    "Open": "Apertura",
    "High": "Maximo",
    "Low": "Minimo",
    "Close": "Cierre",
    "Volume": "Volumen",
    "Dividends": "Dividendos",
    "Stock Splits": "Division_Acciones"
})

print("Columnas traducidas:", df.columns.tolist())

In [ ]:
# Comprueba que los precios financieros sean mayores que cero
columnas_precios = ["Apertura", "Maximo", "Minimo", "Cierre"]
conteo_precios_invalidos = (df[columnas_precios] <= 0).sum()

print("Valores menores o iguales a cero por columna:")
display(conteo_precios_invalidos)

# Conserva solamente filas con precios válidos
filas_con_precio_invalido = (df[columnas_precios] <= 0).any(axis=1).sum()
df = (
    df.loc[(df[columnas_precios] > 0).all(axis=1)]
      .reset_index(drop=True)
      .copy()
)

print("Filas con precios inválidos eliminadas:", filas_con_precio_invalido)

In [ ]:
# Identifica columnas que contienen un único valor y no aportan variación
columnas_constantes = [
    columna
    for columna in df.columns
    if df[columna].nunique(dropna=False) <= 1
]

print("Columnas constantes eliminadas:", columnas_constantes)
df = df.drop(columns=columnas_constantes)

print("Dimensiones después de la limpieza:", df.shape)
display(df.head())

## 5. Transformación de datos

Se crean nuevas columnas mediante operaciones vectorizadas. El retorno diario representa la variación porcentual del cierre, mientras que el rango diario muestra la diferencia relativa entre el precio máximo y mínimo.

In [ ]:
# Calcula la variación porcentual respecto al cierre anterior
df["Retorno_Diario"] = df["Cierre"].pct_change(fill_method=None)

# Calcula la amplitud diaria del precio respecto a la apertura
df["Rango_Diario"] = (
    (df["Maximo"] - df["Minimo"]) / df["Apertura"]
)

# Extrae el año para realizar una agrupación cronológica
df["Anio"] = df["Fecha"].dt.year

display(
    df[[
        "Fecha",
        "Apertura",
        "Maximo",
        "Minimo",
        "Cierre",
        "Volumen",
        "Retorno_Diario",
        "Rango_Diario"
    ]].head(10)
)

## 6. Análisis estadístico con Pandas

Las estadísticas descriptivas resumen la distribución, tendencia central y dispersión de las variables numéricas.

In [ ]:
# Genera las estadísticas y transpone la tabla para facilitar su lectura
estadisticas = df.describe().T
display(estadisticas)

In [ ]:
# Resume el comportamiento financiero por año
resumen_anual = (
    df.groupby("Anio")
      .agg(
          Cierre_Promedio=("Cierre", "mean"),
          Volumen_Promedio=("Volumen", "mean"),
          Retorno_Promedio=("Retorno_Diario", "mean")
      )
)

print("Resumen de los últimos diez años disponibles:")
display(resumen_anual.tail(10))

## 7. Operaciones con NumPy

Pandas organiza los datos en un DataFrame. Después de la limpieza, NumPy permite extraer vectores y matrices numéricas de alto rendimiento que posteriormente podrán alimentar un algoritmo de Machine Learning.

In [ ]:
# Convierte los retornos válidos en un vector de NumPy
vector_retornos = df["Retorno_Diario"].dropna().to_numpy()

print("Dimensiones del vector:", vector_retornos.shape)
print("Retorno promedio:", np.mean(vector_retornos))
print("Desviación estándar:", np.std(vector_retornos))
print("Retorno mínimo:", np.min(vector_retornos))
print("Retorno máximo:", np.max(vector_retornos))

In [ ]:
# Extrae una matriz con las características financieras básicas
columnas_matriz = [
    "Apertura",
    "Maximo",
    "Minimo",
    "Cierre",
    "Volumen",
    "Retorno_Diario",
    "Rango_Diario"
]

matriz_financiera = (
    df[columnas_matriz]
    .dropna()
    .to_numpy()
)

print("Dimensiones de la matriz financiera:", matriz_financiera.shape)
print("Primeras cinco filas de la matriz:")
print(matriz_financiera[:5])

## 8. Visualización de datos

Las visualizaciones permiten reconocer la distribución de los retornos, la evolución histórica del índice y las relaciones entre las variables.

In [ ]:
# Muestra la distribución de la variación porcentual diaria
plt.figure(figsize=(10, 5))
sns.histplot(
    data=df.dropna(subset=["Retorno_Diario"]),
    x="Retorno_Diario",
    bins=40,
    kde=True
)

plt.title("Distribución de los retornos diarios del S&P 500")
plt.xlabel("Retorno diario")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

In [ ]:
# Representa la evolución del precio de cierre
plt.figure(figsize=(12, 5))
plt.plot(df["Fecha"], df["Cierre"], color="navy", linewidth=1)

plt.title("Evolución histórica del precio de cierre del S&P 500")
plt.xlabel("Fecha")
plt.ylabel("Precio de cierre")
plt.tight_layout()
plt.show()

In [ ]:
# Calcula y representa la correlación de las variables numéricas relevantes
columnas_correlacion = [
    "Apertura",
    "Maximo",
    "Minimo",
    "Cierre",
    "Volumen",
    "Retorno_Diario",
    "Rango_Diario"
]

matriz_correlacion = df[columnas_correlacion].corr()

plt.figure(figsize=(11, 7))
sns.heatmap(
    matriz_correlacion,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    center=0
)

plt.title("Matriz de correlación de las variables financieras")
plt.tight_layout()
plt.show()

## 9. Conclusiones preliminares

- El dataset fue cargado y organizado en un DataFrame de Pandas.
- Se revisaron valores nulos, duplicados, fechas y precios inválidos.
- La fecha fue convertida a un tipo temporal y las columnas fueron traducidas al español.
- Se eliminaron las columnas constantes porque no aportan variación al análisis.
- Se crearon el retorno y el rango diario mediante operaciones vectorizadas.
- Se extrajeron un vector y una matriz de NumPy para representar numéricamente la información.
- Las visualizaciones permiten examinar la distribución, evolución y correlación de las variables.

En el siguiente avance se definirán las variables independientes (X) y la variable dependiente (y), se dividirán cronológicamente los datos y se entrenará un primer modelo de clasificación con Scikit-learn.